In [0]:
spark.version

'4.1.0'

## Chargement et exploration du dataset Steam

In [0]:
from pyspark.sql import functions as F

# Chargement depuis S3
df_raw = spark.read.json("s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json")

# Aplatissement : on extrait tous les champs de data.*
df = df_raw.select("data.*")

# Aperçu
df.printSchema()


root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- tags: struct (nullable = true)
 |    |-- 1980s: lon

## Exploration générale

In [0]:
print("Nombre de jeux :", df.count())
print("Colonnes :", len(df.columns))
df.show(3, truncate=50)


Nombre de jeux : 55691
Colonnes : 22
+-------+--------------------------------------------------+-----+----------------------+--------+-------------------------------+--------------------------------------------------+------------+--------------------------------------------------+--------------+--------+------------------------+--------------------+--------+-----+------------------------+------------+------------+--------------------------------------------------+--------------------------------------------------+----+-------+
|  appid|                                        categories|  ccu|             developer|discount|                          genre|                                      header_image|initialprice|                                         languages|          name|negative|                  owners|           platforms|positive|price|               publisher|release_date|required_age|                                 short_description|                                  

In [0]:
from pyspark.sql import functions as F

nulls = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["name", "developer", "publisher", "genre", "release_date",
              "price", "initialprice", "discount", "owners",
              "positive", "negative", "required_age", "languages"]
])
nulls.show()


+----+---------+---------+-----+------------+-----+------------+--------+------+--------+--------+------------+---------+
|name|developer|publisher|genre|release_date|price|initialprice|discount|owners|positive|negative|required_age|languages|
+----+---------+---------+-----+------------+-----+------------+--------+------+--------+--------+------------+---------+
|   0|        0|        0|    0|           0|    0|           0|       0|     0|       0|       0|           0|        0|
+----+---------+---------+-----+------------+-----+------------+--------+------+--------+--------+------------+---------+



## Nettoyage et préparation des données


In [0]:
def cast_numeric(col_name):
    return F.when(F.col(col_name) == "", None).otherwise(F.col(col_name).cast("double"))

df_clean = df.withColumn("prix_euros", cast_numeric("price") / 100) \
             .withColumn("prix_initial_euros", cast_numeric("initialprice") / 100) \
             .withColumn("discount_pct", cast_numeric("discount")) \
             .withColumn("age_requis", F.when(F.col("required_age") == "", None).otherwise(F.col("required_age").cast("int"))) \
             .withColumn("annee_sortie", F.when(F.col("release_date") == "", None).otherwise(F.substring("release_date", 1, 4).cast("int"))) \
             .withColumn("ratio_positif",
                         F.round(F.col("positive") / (F.col("positive") + F.col("negative") + 1) * 100, 1))

df_clean = df_clean.withColumn("windows", F.col("platforms.windows").cast("int")) \
                   .withColumn("mac", F.col("platforms.mac").cast("int")) \
                   .withColumn("linux", F.col("platforms.linux").cast("int"))

df_clean.select("name", "prix_euros", "discount_pct", "annee_sortie", "ratio_positif", "windows", "mac", "linux").show(5)


+--------------------+----------+------------+------------+-------------+-------+---+-----+
|                name|prix_euros|discount_pct|annee_sortie|ratio_positif|windows|mac|linux|
+--------------------+----------+------------+------------+-------------+-------+---+-----+
|      Counter-Strike|      9.99|         0.0|        2000|         97.5|      1|  1|    1|
|           ASCENXION|      9.99|         0.0|        2021|         81.8|      1|  0|    0|
|         Crown Trick|      5.99|        70.0|        2020|         86.2|      1|  0|    0|
|Cook, Serve, Deli...|     19.99|         0.0|        2020|         93.1|      1|  1|    0|
|            细胞战争|      1.99|         0.0|        2019|          0.0|      1|  0|    0|
+--------------------+----------+------------+------------+-------------+-------+---+-----+
only showing top 5 rows


## Analyse macro
### Quels publishers ont sorti le plus de jeux sur Steam ?


In [0]:
top_publishers = df_clean.groupBy("publisher") \
    .agg(F.count("appid").alias("nb_jeux")) \
    .orderBy(F.desc("nb_jeux")) \
    .limit(20)

display(top_publishers)


publisher,nb_jeux
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141
Choice of Games,140
HH-Games,132
Sekai Project,132
,132
Ubisoft,127


Databricks visualization. Run in Databricks to view.

**Observation :** Big Fish Games domine largement avec 422 jeux, suivi de 8floor (202) et SEGA (165).
On note une ligne vide (publisher inconnu, 132 jeux) à filtrer si besoin.
**Ubisoft** apparaît en 10ème position avec 127 jeux — présence significative mais loin des spécialistes du volume.
Les leaders sont surtout des éditeurs de jeux casual/indépendants prolifiques, pas nécessairement les plus populaires.

### Quels sont les jeux les mieux notés ?


In [0]:
# Minimum 100 avis pour filtrer les jeux avec peu de votes
jeux_bien_notes = df_clean.filter((F.col("positive") + F.col("negative")) >= 100) \
    .select("name", "publisher", "positive", "negative", "ratio_positif") \
    .orderBy(F.desc("ratio_positif")) \
    .limit(20)

display(jeux_bien_notes)


name,publisher,positive,negative,ratio_positif
Flowers -Le volume sur ete-,JAST USA,937,1,99.8
The Void Rains Upon Her Heart,The Hidden Levels,496,0,99.8
祈風 Inorikaze,觀象草圖 Astrolabe Draft,327,0,99.7
秘封旅行 ~ Secret Sealing Travel,鸽屋谷,218,0,99.5
Elasto Mania Remastered,Elasto Mania Team,190,0,99.5
Freshly Frosted,The Quantum Astrophysicists Guild,157,0,99.4
HAYAI,Chaoclypse,148,0,99.3
Cleo - a pirate's tale,Greycap Audiovisual Mediadesign UG,407,2,99.3
The Upturned,Zeekerss,454,2,99.3
Aseprite,Igara Studio,11823,80,99.3


**Observation :** Les jeux les mieux notés (>99%) sont majoritairement des jeux indépendants de niche —
visual novels japonais, puzzles, jeux courts soignés. Aseprite (outil de pixel art, 99.3%, 11 823 avis)
et A Short Hike (99.2%, 11 645 avis) sont les titres grand public les mieux notés.
La corrélation entre taille d'audience et note parfaite est faible : les AAA n'apparaissent pas ici.


### Y a-t-il des années avec plus de sorties ? Impact du Covid ?


In [0]:
sorties_par_an = df_clean.filter(
    (F.col("annee_sortie") >= 2000) & (F.col("annee_sortie") <= 2023)
).groupBy("annee_sortie") \
 .agg(F.count("appid").alias("nb_sorties")) \
 .orderBy("annee_sortie")

display(sorties_par_an)


annee_sortie,nb_sorties
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61
2007,98
2008,159
2009,311


Databricks visualization. Run in Databricks to view.

**Observation :** La croissance des sorties est exponentielle à partir de 2014, coïncidant avec
l'ouverture de Steam Greenlight (2012) puis Steam Direct (2017) qui a drastiquement simplifié la publication.
**Effet Covid (2020) :** contrairement à ce qu'on pourrait attendre, les sorties ont *augmenté* en 2020
(8 305 jeux, +19% vs 2019). Le confinement a stimulé à la fois la consommation et le développement de jeux.
2021 est l'année record avec 8 823 sorties. 2022 marque un léger recul, probablement lié au retour à la normale.


### Distribution des prix et des remises


In [0]:
# Distribution des prix (jeux payants uniquement, prix < 100€ pour exclure les outliers)
prix_dist = df_clean.filter((F.col("prix_euros") > 0) & (F.col("prix_euros") < 100)) \
    .select("prix_euros")

display(prix_dist)


prix_euros
9.99
9.99
5.99
19.99
1.99
7.99
12.99
2.99
13.99
0.99


Databricks visualization. Run in Databricks to view.

In [0]:
# Stats sur les remises
df_clean.select(
    F.round(F.avg("prix_euros"), 2).alias("prix_moyen"),
    F.round(F.avg(F.when(F.col("discount_pct") > 0, F.col("discount_pct"))), 1).alias("remise_moyenne_si_remisee"),
    F.round(F.sum(F.when(F.col("discount_pct") > 0, 1).otherwise(0)) / F.count("*") * 100, 1).alias("pct_jeux_avec_remise"),
    F.round(F.sum(F.when(F.col("prix_euros") == 0, 1).otherwise(0)) / F.count("*") * 100, 1).alias("pct_jeux_gratuits")
).show()


+----------+-------------------------+--------------------+-----------------+
|prix_moyen|remise_moyenne_si_remisee|pct_jeux_avec_remise|pct_jeux_gratuits|
+----------+-------------------------+--------------------+-----------------+
|      7.73|                     57.6|                 4.5|             14.0|
+----------+-------------------------+--------------------+-----------------+



In [0]:
df_clean.select(
    F.round(F.avg("prix_euros"), 2).alias("prix_moyen"),
    F.round(F.avg(F.when(F.col("discount_pct") > 0, F.col("discount_pct"))), 1).alias("remise_moyenne_si_remisee"),
    F.round(F.sum(F.when(F.col("discount_pct") > 0, 1).otherwise(0)) / F.count("*") * 100, 1).alias("pct_jeux_avec_remise"),
    F.round(F.sum(F.when(F.col("prix_euros") == 0, 1).otherwise(0)) / F.count("*") * 100, 1).alias("pct_jeux_gratuits")
).show()


+----------+-------------------------+--------------------+-----------------+
|prix_moyen|remise_moyenne_si_remisee|pct_jeux_avec_remise|pct_jeux_gratuits|
+----------+-------------------------+--------------------+-----------------+
|      7.73|                     57.6|                 4.5|             14.0|
+----------+-------------------------+--------------------+-----------------+



**Observation :** Le prix moyen d'un jeu Steam est **7,73€**, reflet d'un catalogue très orienté indie à petits prix.
**14%** des jeux sont gratuits (free-to-play). Parmi les jeux remisés au moment du snapshot (seulement **4,5%**),
la remise moyenne est massive : **57,6%**. Les promotions Steam sont donc rares instantanément mais très agressives
quand elles ont lieu. La grande majorité des jeux est vendue au prix catalogue.


### Quelles sont les langues les plus représentées ?

In [0]:
from pyspark.sql.functions import split, explode, trim

# Explosion des langues (séparées par ", ")
df_langues = df_clean.select(
    explode(split(F.col("languages"), ",")).alias("langue")
).withColumn("langue", trim(F.col("langue"))) \
 .filter(F.col("langue") != "")

top_langues = df_langues.groupBy("langue") \
    .agg(F.count("*").alias("nb_jeux")) \
    .orderBy(F.desc("nb_jeux")) \
    .limit(15)

display(top_langues)


langue,nb_jeux
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600


Databricks visualization. Run in Databricks to view.

**Observation :** L'anglais est quasi-universel (55 116 jeux sur 55 691, soit ~99%).
L'allemand (14 019), le français (13 426) et le russe (12 922) constituent le podium européen.
Le chinois simplifié (12 782) témoigne du poids croissant du marché asiatique sur Steam.
Le japonais (10 368) reflète l'influence des studios indépendants japonais.
Le portugais brésilien (6 750) dépasse le portugais Portugal (4 011), confirmant le Brésil comme marché prioritaire.


### Jeux interdits aux moins de 16/18 ans


In [0]:
df_clean = df.withColumn("prix_euros", cast_numeric("price") / 100) \
             .withColumn("prix_initial_euros", cast_numeric("initialprice") / 100) \
             .withColumn("discount_pct", cast_numeric("discount")) \
             .withColumn("age_requis", F.when(
                 F.col("required_age").rlike("^[0-9]+$"), F.col("required_age").cast("int")
             ).otherwise(None)) \
             .withColumn("annee_sortie", F.when(F.col("release_date") == "", None).otherwise(F.substring("release_date", 1, 4).cast("int"))) \
             .withColumn("ratio_positif",
                         F.round(F.col("positive") / (F.col("positive") + F.col("negative") + 1) * 100, 1))

df_clean = df_clean.withColumn("windows", F.col("platforms.windows").cast("int")) \
                   .withColumn("mac", F.col("platforms.mac").cast("int")) \
                   .withColumn("linux", F.col("platforms.linux").cast("int"))

print("OK")


OK


In [0]:
age_dist = df_clean.groupBy("age_requis") \
    .agg(F.count("*").alias("nb_jeux")) \
    .orderBy("age_requis")

display(age_dist)


age_requis,nb_jeux
null,3
0,55030
3,3
5,1
6,4
7,2
8,3
9,1
10,7
12,32


**Observation :** La quasi-totalité des jeux (55 030 sur 55 691, soit **98,8%**) n'ont aucune restriction d'âge (0).
Seuls **223 jeux** sont classés 18+ et **38 jeux** 16+. Les valeurs aberrantes (35, 180) sont des erreurs de saisie
à exclure. Steam est donc une plateforme très largement accessible à tous publics, les jeux adultes
représentant une part marginale du catalogue.


## Analyse par genres
### Quels sont les genres les plus représentés ?


In [0]:
df_genres = df_clean.select(
    explode(split(F.col("genre"), ",")).alias("genre_unique")
).withColumn("genre_unique", trim(F.col("genre_unique"))) \
 .filter(F.col("genre_unique") != "")

top_genres = df_genres.groupBy("genre_unique") \
    .agg(F.count("*").alias("nb_jeux")) \
    .orderBy(F.desc("nb_jeux")) \
    .limit(15)

display(top_genres)


genre_unique,nb_jeux
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

**Observation :** Le genre **Indie** domine massivement avec 39 681 jeux (71% du catalogue) — reflet de
l'ouverture de Steam aux développeurs indépendants via Steam Direct. Action (23 759) et Casual (22 086)
complètent le podium. On note que "Early Access" et "Free to Play" apparaissent comme genres dans le dataset,
ce qui sont en réalité des modèles de distribution plutôt que des genres — à garder en tête pour l'interprétation.
Les genres Utilities et Design & Illustration confirment que Steam héberge aussi des logiciels non-jeux.


### Quel genre a le meilleur ratio de critiques positives ?


In [0]:
# Jointure genre × reviews (min 50 avis pour fiabilité)
df_genre_reviews = df_clean.filter(
    (F.col("positive") + F.col("negative")) >= 50
).select(
    explode(split(F.col("genre"), ",")).alias("genre_unique"),
    "ratio_positif"
).withColumn("genre_unique", trim(F.col("genre_unique"))) \
 .filter(F.col("genre_unique") != "")

ratio_par_genre = df_genre_reviews.groupBy("genre_unique") \
    .agg(
        F.round(F.avg("ratio_positif"), 1).alias("ratio_moyen"),
        F.count("*").alias("nb_jeux")
    ) \
    .filter(F.col("nb_jeux") >= 50) \
    .orderBy(F.desc("ratio_moyen"))

display(ratio_par_genre)


genre_unique,ratio_moyen,nb_jeux
Game Development,83.0,50
Design & Illustration,78.9,133
Animation & Modeling,78.0,121
Adventure,77.7,8766
Casual,77.5,6847
Indie,77.3,14570
RPG,76.2,4551
Utilities,75.8,216
Action,75.6,9190
Strategy,74.4,4704


Databricks visualization. Run in Databricks to view.

**Observation :** Les outils créatifs (Game Development, Design & Illustration, Animation & Modeling)
obtiennent les meilleurs ratios (~78-83%) — leur public est ciblé et satisfait. Parmi les vrais genres de jeux,
**Adventure** (77,7%) et **Casual** (77,5%) sont les mieux notés.
**Massively Multiplayer** (65,3%) et **Violent** (65,9%) obtiennent les scores les plus faibles —
les MMO souffrent souvent de problèmes de serveurs et de monétisation aggressive qui polarisent les avis.
Les écarts restent modérés (65-78%), ce qui suggère que la qualité est relativement homogène entre genres.


### Les publishers ont-ils des genres favoris ?


In [0]:
# Top 10 publishers × genres (sur les publishers avec le plus de jeux)
top_pub = [r["publisher"] for r in df_clean.groupBy("publisher")
           .agg(F.count("*").alias("n"))
           .orderBy(F.desc("n")).limit(10).collect()]

df_pub_genre = df_clean.filter(F.col("publisher").isin(top_pub)) \
    .select("publisher", explode(split(F.col("genre"), ",")).alias("genre_unique")) \
    .withColumn("genre_unique", trim(F.col("genre_unique"))) \
    .filter(F.col("genre_unique") != "")

pub_genre = df_pub_genre.groupBy("publisher", "genre_unique") \
    .agg(F.count("*").alias("nb_jeux")) \
    .orderBy("publisher", F.desc("nb_jeux"))

display(pub_genre)


publisher,genre_unique,nb_jeux
,Indie,106
,Action,70
,Casual,64
,Adventure,37
,Simulation,23
,Strategy,19
,RPG,16
,Early Access,11
,Racing,8
,Sports,6


Databricks visualization. Run in Databricks to view.

**Observation :** Les spécialisations par publisher sont très marquées :
- **Big Fish Games** : quasi-exclusivement Casual + Adventure (hidden object games)
- **8floor** : Casual à 90%
- **Choice of Games** : RPG + Adventure (visual novels textuels)
- **HH-Games** : Casual dominant
- **SEGA** : portfolio diversifié, Action en tête
- **Square Enix** : Action + RPG à parité, fidèle à son ADN JRPG
- **Ubisoft** : Action dominant (70 jeux), suivi d'Adventure — cohérent avec ses franchises (Assassin's Creed, Far Cry)


## Analyse par plateformes
### Répartition Windows / Mac / Linux


In [0]:
total = df_clean.count()

plateformes = df_clean.select(
    F.round(F.sum("windows") / total * 100, 1).alias("pct_windows"),
    F.round(F.sum("mac") / total * 100, 1).alias("pct_mac"),
    F.round(F.sum("linux") / total * 100, 1).alias("pct_linux")
)
plateformes.show()

# Nombre de jeux exclusifs Windows
exclusif_windows = df_clean.filter(
    (F.col("windows") == 1) & (F.col("mac") == 0) & (F.col("linux") == 0)
).count()
print(f"Exclusifs Windows : {exclusif_windows} ({round(exclusif_windows/total*100,1)}%)")


+-----------+-------+---------+
|pct_windows|pct_mac|pct_linux|
+-----------+-------+---------+
|      100.0|   22.9|     15.2|
+-----------+-------+---------+

Exclusifs Windows : 41271 (74.1%)


**Observation :** Windows domine sans partage : **100%** des jeux Steam sont disponibles sur Windows.
Seuls **22,9%** supportent Mac et **15,2%** Linux. **74,1%** des jeux sont exclusifs Windows —
les développeurs indépendants, qui constituent la majorité du catalogue, ciblent en priorité la plateforme
la plus répandue. Mac et Linux restent des cibles secondaires, souvent ajoutées après le lancement Windows.


### Certains genres sont-ils préférentiellement disponibles sur Mac ou Linux ?


In [0]:
df_genre_plat = df_clean.select(
    explode(split(F.col("genre"), ",")).alias("genre_unique"),
    "windows", "mac", "linux"
).withColumn("genre_unique", trim(F.col("genre_unique"))) \
 .filter(F.col("genre_unique") != "")

genre_plat = df_genre_plat.groupBy("genre_unique") \
    .agg(
        F.count("*").alias("nb_jeux"),
        F.round(F.avg("mac") * 100, 1).alias("pct_mac"),
        F.round(F.avg("linux") * 100, 1).alias("pct_linux")
    ) \
    .filter(F.col("nb_jeux") >= 100) \
    .orderBy(F.desc("pct_mac"))

display(genre_plat)


genre_unique,nb_jeux,pct_mac,pct_linux
Game Development,159,32.7,22.0
Strategy,10895,27.6,16.8
Indie,39681,25.0,17.6
Free to Play,3393,24.9,14.0
Design & Illustration,406,24.6,13.3
RPG,9534,23.6,16.0
Adventure,21431,23.5,15.4
Casual,22086,23.2,15.0
Animation & Modeling,322,23.0,11.8
Simulation,10836,22.5,14.1


Databricks visualization. Run in Databricks to view.

**Observation :** Les genres les mieux supportés sur Mac sont **Game Development** (32,7%), **Strategy** (27,6%)
et **Indie** (25%) — des genres historiquement populaires chez les utilisateurs Mac créatifs.
À l'inverse, **Video Production** (11,7%), **Photo Editing** (15,2%) et **Utilities** (15%) ont les taux Mac
les plus faibles malgré leur nature créative, probablement car leurs alternatives natives macOS sont déjà très fortes.
**Linux** suit globalement les mêmes tendances que Mac mais avec des taux inférieurs (~5-10 points de moins).
Les jeux **Action** et **Sports** sont les moins bien portés hors Windows (19-20% Mac).


## Synthèse et recommandations pour Ubisoft


### Principaux enseignements

**Marché global :**
- 55 691 jeux sur Steam, dominé à 71% par l'Indie — le marché est saturé en volume mais pas en qualité.
- Les sorties ont explosé depuis 2014 (Steam Direct) et atteint un pic en 2021 (8 823 jeux). La compétition est maximale.
- Prix moyen : 7,73€. Ubisoft (AAA, 30-70€) évolue dans un segment premium distinct.

**Genres porteurs :**
- **Adventure** et **Casual** obtiennent les meilleurs ratios de satisfaction (>77%).
- **Action** reste le genre le plus publié par les grands studios (SEGA, Square Enix, Ubisoft).
- Les MMO souffrent d'une satisfaction plus faible (65%) — risque à anticiper.

**Plateformes :**
- Windows incontournable (100%). Mac et Linux sont des extensions secondaires (~23% / ~15%).
- Investir dans un port Mac/Linux bien fait peut différencier un titre sur des genres comme Strategy ou RPG.

**Recommandations pour le nouveau jeu Ubisoft :**
1. Cibler **Action-Adventure** : genre dominant d'Ubisoft, excellent ratio de satisfaction.
2. Proposer un **prix premium justifié** (29-59€) — le marché discount est saturé par l'Indie.
3. Assurer un **support multilingue large** : anglais + allemand + français + russe + chinois simplifié couvre 90% du marché.
4. Viser une sortie en **période creuse** (hors pic novembre) pour maximiser la visibilité.
5. **Éviter l'Early Access** : ratio de satisfaction inférieur (73,4%) et perçu négativement sur les AAA.
